# More than a Hack 2027 — Colab runner

This notebook runs the possession model where the match bundle lives. The laptop only gets the submission CSV back.

**CPU runtime.** No GPU. Do not change that.

## Paste-back contract

When a run finishes, paste into chat **only**:

1. The alignment lines (`alignment agrees`, record count, `time_base_agrees`).
2. The `validate_submission.py` block (`OK`, label counts, possession count, mean duration).
3. The scoreboard table from the public five-minute check.

Do not paste the download URL, Drive paths, ATD lines, homography rows, Smart Tagging XML, or screenshots of the pitch. Those are match data. This chat is not a place they can go.

The public score is a wiring check. A `spellF1` near zero means the frames are shifted. A plausible score means stop. Do not edit `src/mtah/params.py` to climb that number. On this window `baseline_smooth125` wins, and on the full match it loses to `baseline_smooth50` (spellF1 **0.2433**), which is the floor that matters.

## Before the first run

1. In Colab: Secrets (the key icon) → add `MTAH_DATA_URL` = the organiser download link. Direct file URL, not a preview page.
2. Upload this repository to Google Drive at `MyDrive/MoreThanAHack/repo`. The folder must contain `scripts/`, `src/`, `evaluation/`, `starter/`, and `games.py`.
3. Do not upload `data/`, either video, or the zip. Free Drive cannot hold the 13GB bundle.

Part A downloads the zip onto the Colab disk, keeps four tactical files, and deletes the archive. Later runs skip the download.


## Setup

Mount Drive, copy the code onto the Colab disk, install `numpy` and `pandas`. Leave `REPO_URL` empty until this project has a public GitHub remote. Then set it and this cell will `git clone` instead of copying from Drive.


In [ ]:
import os
import shutil
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

# Public git URL, once it exists. Empty means "copy MyDrive/MoreThanAHack/repo".
REPO_URL = ""
DRIVE_CODE = "/content/drive/MyDrive/MoreThanAHack/repo"
DEST = "/content/morethanahack"
DATA_DIR = "/content/drive/MyDrive/MoreThanAHack/data"
os.environ["MTAH_DATA_DIR"] = DATA_DIR


def _ignore(directory, names):
    if os.path.abspath(directory) == os.path.abspath(DRIVE_CODE):
        return [name for name in names if name in ("data", ".git")]
    return [name for name in names if name.endswith(".mp4")]


if REPO_URL:
    if os.path.isdir(os.path.join(DEST, ".git")):
        subprocess.check_call(["git", "-C", DEST, "pull", "--ff-only"])
    else:
        if os.path.isdir(DEST):
            shutil.rmtree(DEST)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, DEST])
else:
    if not os.path.isdir(os.path.join(DRIVE_CODE, "scripts")):
        raise SystemExit(
            "Upload this project to MyDrive/MoreThanAHack/repo "
            "(scripts/, src/, evaluation/, starter/, games.py). "
            "Do not upload the match bundle."
        )
    shutil.copytree(DRIVE_CODE, DEST, dirs_exist_ok=True, ignore=_ignore)

os.chdir(DEST)
sys.path.insert(0, DEST)
sys.path.insert(0, os.path.join(DEST, "src"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("repo ready")
print("data dir set (files checked in the next cell)")


## Part A — fetch the four tactical files, once

Wanted files:

- `BAR-ATH_tactical_atd.txt`
- `BAR-ATH_tactical_atd.xml`
- `BAR-ATH_tactical_homography.csv`
- `BAR-ATH_tactical_smart_tagging.xml`

The zip stays on the Colab disk. Videos inside it are never written to Drive. After a successful extract the zip is deleted. If this cell says the files are already on Drive, leave them there.


In [ ]:
import os
import subprocess

from google.colab import userdata

from mtah.fetch import WANTED, extract_archive

TACTICAL = os.path.join(DATA_DIR, "tactical")
ARCHIVE = "/content/mtah_bundle"


def _ready(name):
    path = os.path.join(TACTICAL, name)
    return os.path.isfile(path) and os.path.getsize(path) > 0


if all(_ready(name) for name in WANTED):
    print("tactical files already on Drive; bundle download skipped")
    for name in WANTED:
        print("{}  {} bytes".format(name, os.path.getsize(os.path.join(TACTICAL, name))))
else:
    try:
        url = userdata.get("MTAH_DATA_URL")
    except Exception:
        raise SystemExit(
            "Add a Colab secret named MTAH_DATA_URL. "
            "Do not paste the URL into this notebook or into chat."
        )
    if not url or not str(url).startswith("http"):
        raise SystemExit("MTAH_DATA_URL must be an http(s) link to the bundle file.")
    print("downloading the bundle onto the Colab disk")
    download = subprocess.run(
        ["wget", "-q", "-c", "--tries", "3", "-O", ARCHIVE, url],
        capture_output=True,
    )
    if download.returncode != 0:
        raise SystemExit(
            "bundle download failed (exit {}). Check the secret URL. It was not printed.".format(
                download.returncode
            )
        )
    try:
        written = extract_archive(ARCHIVE, TACTICAL)
    except Exception:
        print("extract failed. The archive is still at /content/mtah_bundle.")
        print("Delete that file before retrying if the disk is full.")
        raise
    os.remove(ARCHIVE)
    print("deleted the local archive. Videos were not written to Drive.")
    for name, size in written:
        print("{}  {} bytes".format(name, size))


## Part B — alignment

Counts and pass/fail only. A mismatch here means stop. Do not invent a frame offset.


In [ ]:
from starter.load_atd import check_alignment_raw
from starter.load_smart_tagging import check_time_base, load

report = check_alignment_raw(data_dir=DATA_DIR)
print("alignment agrees:", report["agrees"])
print("records", report["n_records"], "video_frames", report["n_video_frames"])
print("frame_id_range", report["frame_id_range"], "contiguous", report["contiguous"])
if not report["agrees"]:
    raise SystemExit("ATD is not one record per tactical frame numbered from 1. Stop.")

events = load(data_dir=DATA_DIR)
time_base = check_time_base(events)
print("smart_tagging_instances", len(events))
print("time_base_agrees", time_base["agrees"])
for row in time_base["halves"]:
    print(row["half"], "kickoff", row["true_kickoff_frame"], "contains", row["contains_kickoff"])
if not time_base["agrees"]:
    raise SystemExit("Smart Tagging clock does not contain the published kickoffs. Stop.")


## Part B — build and validate

Writes `evaluation/results/submission.csv` (199,319 rows). Validation checks the file shape and prints how many possessions the file claims. That possession count is the number to read. A file of two-second scraps is flicker.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "scripts/build_submission.py",
    "--data-dir", DATA_DIR,
    "--out", "evaluation/results/submission.csv",
])
print("---")
subprocess.check_call([
    sys.executable, "evaluation/validate_submission.py",
    "evaluation/results/submission.csv",
])


## Part B — public five-minute wiring check

This scores every CSV in `evaluation/results/`, including the four shipped baselines, against the **public** 17 possessions. It does not score the full match.

Read it once. If our `spellF1` is near zero, the index is wrong. If it is in a normal range, do not change `src/mtah/params.py` to beat `baseline_smooth125` on this table. That baseline is the one that loses on the match we are actually ranked on.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "evaluation/run_evaluation.py",
    "--results", "evaluation/results",
    "--no-detail",
])


## Download the CSV

Take only `submission.csv` back to the laptop (~1.9MB). Then, from the repo:

```bash
python evaluation/validate_submission.py path/to/submission.csv
python evaluation/run_evaluation.py --results path/to/the_folder --no-detail
```

The folder you point `--results` at should contain that one CSV, or the scorer will also rank whatever else is in there.


In [ ]:
from google.colab import files

files.download("evaluation/results/submission.csv")
print("PASTE BACK TO CHAT, and nothing else:")
print("  alignment agrees / record count / time_base_agrees")
print("  the validate_submission.py block")
print("  the scoreboard table")
print("Do not paste the download URL, tracking rows, or this notebook's secret.")
